# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I am framing the Refresh / Content Opportunity Scoring lane as a scoring problem. For each content item, I would estimate a priority score for refresh review, and then use that score to rank the queue of pages an editor should inspect first. The model is trained against an observed decline proxy, so the output is decision-support rather than a guarantee.

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    data_path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
    if data_path.exists():
        break
    data_path = None

if data_path is None:
    raise FileNotFoundError("Could not find the starter dataset from the current notebook location.")

# Load the starter slice for the refresh lane.
df = pd.read_csv(data_path)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
print("Declining share:", round(df["is_declining_label"].mean(), 3))
print(df["is_declining_label"].value_counts().to_dict())

SyntaxError: invalid syntax (3381007814.py, line 21)

## 2. Target or proxy

The target I would predict is a probability-like score for whether a content item is declining, using an observed proxy from the data: whether its recent impression trend is labeled as "down" based on the change between the last 30 days and the previous 30 days. That is an observed outcome in the dataset, not a hand-authored business rule, although it is still a proxy for the broader idea of "needs refresh."

In [ ]:
# Preview the proxy target alongside the raw trend fields.
target_preview = df[["content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"]].head(10)
display(target_preview)

## 3. Success metric

I would defend Precision@50 for the top 50 content items ranked by the model score. If the model sends an editor to 50 pages first, a high Precision@50 means the queue is genuinely concentrated on items that are actually declining and worth reviewing. That metric fits the action of triaging a limited review budget.

In [ ]:
# A simple sanity check for the target distribution before any modeling.
positive_rate = df["is_declining_label"].mean()
print(f"Base-rate of declining proxy: {positive_rate:.3f}")
print("A model that beats this rate on Precision@50 would be useful for review triage.")

## 4. The unit of analysis, as a real dataframe

In this starter slice, one row is one content item (one page/article). The dataframe therefore represents a set of candidate pages that could be reviewed for refresh, with each row carrying content metadata, traffic history, and the observed decline proxy.

In [ ]:
# Show the unit of analysis directly.
unit_of_analysis = df[["content_id", "client_id", "content_type", "word_count", "impressions_90d", "trend_direction", "is_declining_label"]].head(8)
display(unit_of_analysis)

## 5. Why ML beats a fixed rule here

A fixed rule would struggle because decline is not driven by one obvious threshold. The signal is a mixture of traffic volume, recent trend, freshness, content type, search intent, position, and client-specific behavior. Those relationships are noisy, non-linear, and partly missing, so a hand-written if-statement would either miss many true opportunities or over-select pages that look similar on paper but are not actually declining. An ML model can learn a weighted combination of these signals and adapt them to the data.

In [ ]:
# A final quick check: the action is to rank a review queue, not to make a deterministic rule.
print("This framing supports an editor action: review the highest-scoring pages first.")
print("The model output is a score for triage, and the label is an observed proxy for decline.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.